# Objetivo

Uma loja de produtos culturais e entretenimento, que vende filmes, livros e jogos, planeja criar um espaço dedicado a filmes lançados entre 2000 e 2026. Para decidir quais títulos incluir, utilizará dados do IMDb e selecionará até 100 filmes em cada faixa de lançamento: 2000 a 2009, 2010 a 2019 e 2020 a 2026. Serão considerados filmes com nota média igual ou superior a 7,0, priorizados pelo número de votos recebidos. A seleção servirá como ponto de partida para organizar o novo espaço da loja.

**Perguntas de negócio**

1. Quais são os até 100 filmes com mais votos no IMDb e nota média igual ou superior a 7,0 em cada faixa de lançamento?

2. Quais gêneros aparecem com maior frequência entre os filmes selecionados em cada faixa?

In [0]:
caminho_basics = "/Volumes/mvp_filmes_imdb/default/imdb_raw/title.basics.tsv.gz"

filmes = spark.read.csv(
caminho_basics,
sep="\t",
header=True)

display(filmes.limit(5))

caminho_ratings = '/Volumes/mvp_filmes_imdb/default/imdb_raw/title.ratings.tsv.gz'

ratings = spark.read.csv(
caminho_ratings,
sep="\t",
header=True)
display(ratings.limit(5))

filmes_com_notas = filmes.join (ratings, on="tconst", how="inner")
display(filmes_com_notas.limit(5))

tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
tt0000001,short,Carmencita,Carmencita,0,1894,\N,1,"Documentary,Short"
tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892,\N,5,"Animation,Short"
tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892,\N,5,"Animation,Comedy,Romance"
tt0000004,short,Un bon bock,Un bon bock,0,1892,\N,12,"Animation,Short"
tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893,\N,1,Short


tconst,averageRating,numVotes
tt0000001,5.7,2231
tt0000002,5.4,326
tt0000003,6.4,2386
tt0000004,5.0,202
tt0000005,6.2,3097


tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes
tt0000001,short,Carmencita,Carmencita,0,1894,\N,1,"Documentary,Short",5.7,2231
tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892,\N,5,"Animation,Short",5.4,326
tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892,\N,5,"Animation,Comedy,Romance",6.4,2386
tt0000004,short,Un bon bock,Un bon bock,0,1892,\N,12,"Animation,Short",5.0,202
tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893,\N,1,Short,6.2,3097


In [0]:
display(filmes_com_notas.filter("titleType = 'movie'").limit(5))

filmes_com_generos = filmes_com_notas.filter(filmes_com_notas.titleType == 'movie')
filmes_com_generos = filmes_com_generos.filter(filmes_com_generos.genres != r'\N')
display(filmes_com_generos.limit(5))

from pyspark.sql import functions as F

filmes_2000_2026 = filmes_com_generos.filter(
F.expr("try_cast(startYear AS INT)").between(2000, 2026))

display(filmes_2000_2026.limit(5))

from pyspark.sql import functions as F

filmes_nota_7 = filmes_2000_2026.filter(
 F.expr("try_cast(averageRating AS DOUBLE)") >= 7.0)

display(filmes_nota_7.limit(5).select(
"primaryTitle", "startYear", "genres", "averageRating", "numVotes"))

filmes_nota_7.printSchema()


tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes
tt0000009,movie,Miss Jerry,Miss Jerry,0,1894,\N,45,Romance,5.3,239
tt0000147,movie,The Corbett-Fitzsimmons Fight,The Corbett-Fitzsimmons Fight,0,1897,\N,100,"Documentary,News,Sport",5.3,613
tt0000502,movie,Bohemios,Bohemios,0,1905,\N,100,\N,3.9,28
tt0000574,movie,The Story of the Kelly Gang,The Story of the Kelly Gang,0,1906,\N,70,"Action,Adventure,Biography",6.0,1092
tt0000591,movie,The Prodigal Son,L'enfant prodigue,0,1907,\N,90,Drama,5.0,42


tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes
tt0000009,movie,Miss Jerry,Miss Jerry,0,1894,\N,45,Romance,5.3,239
tt0000147,movie,The Corbett-Fitzsimmons Fight,The Corbett-Fitzsimmons Fight,0,1897,\N,100,"Documentary,News,Sport",5.3,613
tt0000574,movie,The Story of the Kelly Gang,The Story of the Kelly Gang,0,1906,\N,70,"Action,Adventure,Biography",6.0,1092
tt0000591,movie,The Prodigal Son,L'enfant prodigue,0,1907,\N,90,Drama,5.0,42
tt0000615,movie,Robbery Under Arms,Robbery Under Arms,0,1907,\N,\N,Drama,3.7,38


tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes
tt0035423,movie,Kate & Leopold,Kate & Leopold,0,2001,\N,118,"Comedy,Fantasy,Romance",6.4,94812
tt0062336,movie,The Tango of the Widower and Its Distorting Mirror,El tango del viudo y su espejo deformante,0,2020,\N,70,"Drama,Mystery",6.3,299
tt0065414,movie,Appunti per un romanzo sull'immondezza,Appunti per un romanzo sull'immondezza,0,2022,\N,85,Documentary,6.9,41
tt0069049,movie,The Other Side of the Wind,The Other Side of the Wind,0,2018,\N,122,Drama,6.6,8633
tt0070596,movie,Socialist Realism,El realismo socialista,0,2023,\N,78,"Drama,History",6.8,244


primaryTitle,startYear,genres,averageRating,numVotes
"Hamara Shahar - Bombay, Our City",2010,Documentary,8.2,64
West from North Goes South,2004,"Comedy,Mystery",7.7,44
Auf allen Meeren,2002,Documentary,8.0,30
Totschweigen,2026,"Documentary,Drama",8.0,24
Vazir,2015,"Action,Crime,Drama",8.2,73


root
 |-- tconst: string (nullable = true)
 |-- titleType: string (nullable = true)
 |-- primaryTitle: string (nullable = true)
 |-- originalTitle: string (nullable = true)
 |-- isAdult: string (nullable = true)
 |-- startYear: string (nullable = true)
 |-- endYear: string (nullable = true)
 |-- runtimeMinutes: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- averageRating: string (nullable = true)
 |-- numVotes: string (nullable = true)



In [0]:
filmes_com_votos = filmes_nota_7.withColumn("numVotesNumeric",F.expr("try_cast(numVotes AS BIGINT)"))

In [0]:
filmes_2000_2009 = filmes_com_votos.filter(
F.expr("try_cast(startYear AS INT)").between(2000, 2009))

display(filmes_2000_2009.limit(5))
display( filmes_2000_2009.limit(5).select(
['tconst', 'primaryTitle', 'startYear', 'genres', 'averageRating', 'numVotesNumeric']))

filmes_ordenados = filmes_2000_2009.orderBy(
F.col("numVotesNumeric").desc())
display(filmes_ordenados.limit(5).select( 
["tconst", "primaryTitle", "startYear", "genres", "averageRating", "numVotesNumeric"]))
top_100_2000_2009 = filmes_ordenados.limit(100) 
display(top_100_2000_2009.select( 
["tconst", "primaryTitle", "startYear", "genres", "averageRating", "numVotesNumeric"]))



filmes_2010_2019 = filmes_com_votos.filter(
F.expr("try_cast(startYear AS INT)").between(2010,2019))
filmes_ordenados_2010_2019 = filmes_2010_2019.orderBy(
F.col("numVotesNumeric").desc())
top_100_2010_2019 = filmes_ordenados_2010_2019.limit(100) 
display(top_100_2010_2019.select( 
["tconst", "primaryTitle", "startYear", "genres", "averageRating", "numVotesNumeric"]))  
 


filmes_2020_2026 = filmes_com_votos.filter(
F.expr("try_cast(startYear AS INT)").between(2020,2026))
filmes_ordenados_2020_2026 = filmes_2020_2026.orderBy(
F.col("numVotesNumeric").desc())
top_100_2020_2026 = filmes_ordenados_2020_2026.limit(100)
display(top_100_2020_2026.select(
["tconst", "primaryTitle", "startYear", "genres", "averageRating", "numVotesNumeric"]))

selecao_final = top_100_2000_2009.unionByName(top_100_2010_2019).unionByName(top_100_2020_2026)
display(selecao_final.count())

tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes,numVotesNumeric
tt0108549,movie,West from North Goes South,West from North Goes South,0,2004,\N,96,"Comedy,Mystery",7.7,44,44
tt0109173,movie,Auf allen Meeren,Auf allen Meeren,0,2002,\N,95,Documentary,8.0,30,30
tt0113086,movie,Florentino y el diablo,Florentino y el diablo,0,2000,\N,\N,Drama,7.0,23,23
tt0114722,movie,3 noches,3 noches,0,2001,\N,105,"Crime,Thriller",7.1,40,40
tt0115937,movie,Consequence,Consequence,0,2000,\N,91,Drama,7.7,16,16


tconst,primaryTitle,startYear,genres,averageRating,numVotesNumeric
tt0108549,West from North Goes South,2004,"Comedy,Mystery",7.7,44
tt0109173,Auf allen Meeren,2002,Documentary,8.0,30
tt0113086,Florentino y el diablo,2000,Drama,7.0,23
tt0114722,3 noches,2001,"Crime,Thriller",7.1,40
tt0115937,Consequence,2000,Drama,7.7,16


tconst,primaryTitle,startYear,genres,averageRating,numVotesNumeric
tt0468569,The Dark Knight,2008,"Crime,Thriller",9.1,3230709
tt0120737,The Lord of the Rings: The Fellowship of the Ring,2001,"Adventure,Drama,Fantasy",8.9,2239211
tt0167260,The Lord of the Rings: The Return of the King,2003,"Adventure,Drama,Fantasy",9.0,2195417
tt0167261,The Lord of the Rings: The Two Towers,2002,"Adventure,Drama,Fantasy",8.8,1985836
tt0172495,Gladiator,2000,"Action,Adventure,Drama",8.5,1883861


In [0]:
catalogo_final = selecao_final
catalogo_final = catalogo_final.select(
["primaryTitle", "startYear", "genres", "averageRating", "numVotesNumeric"])
display(catalogo_final)


In [0]:
contagem_por_titulo = catalogo_final.groupBy("primaryTitle").count()
display(contagem_por_titulo)
contagem_por_titulo.filter(F.col("count") > 1).count()

In [0]:
catalogo_com_lista_de_generos = catalogo_final.withColumn("genresList",
F.split("genres", ","))

catalogo_com_lista_de_generos = catalogo_com_lista_de_generos.drop("genres")
generos_por_filme = catalogo_com_lista_de_generos.withColumn("genre",
F.explode("genresList"))
generos_por_filme = generos_por_filme.drop("genresList")
display(generos_por_filme)

In [0]:
generos_2000_2009 = generos_por_filme.filter(    
 F.expr("try_cast(startYear AS INT)").between(2000, 2009)) 
contagem_generos_2000_2009 = generos_2000_2009.groupBy("genre").count()
contagem_generos_2000_2009 = contagem_generos_2000_2009.orderBy(
F.col ("count").desc())
display(contagem_generos_2000_2009)


generos_2010_2019 = generos_por_filme.filter(
F.expr("try_cast(startYear AS INT)").between(2010,2019))
contagem_generos_2010_2019 = generos_2010_2019.groupBy("genre").count()
contagem_generos_2010_2019 = contagem_generos_2010_2019.orderBy(
F.col("count").desc())
display(contagem_generos_2010_2019)


generos_2020_2026 = generos_por_filme.filter(
F.expr("try_cast(startYear AS INT)").between(2020,2026))
contagem_generos_2020_2026 = generos_2020_2026.groupBy("genre").count()
contagem_generos_2020_2026 = contagem_generos_2020_2026.orderBy(
F.col("count").desc())
display(contagem_generos_2020_2026)


generos_2000_2009_com_periodo = contagem_generos_2000_2009.withColumn("periodo",
F.lit("2000-2009"))
generos_2010_2019_com_periodo = contagem_generos_2010_2019.withColumn("periodo",
F.lit("2010-2019"))
generos_2020_2026_com_periodo = contagem_generos_2020_2026.withColumn("periodo",
F.lit("2020-2026"))
tabela_generos_final = generos_2000_2009_com_periodo.unionByName(generos_2010_2019_com_periodo).unionByName(generos_2020_2026_com_periodo)
display(tabela_generos_final)

In [0]:
tabela_generos_final.write.mode("overwrite").saveAsTable(
"mvp_filmes_imdb.default.frequencia_generos")

catalogo_final.write.mode("overwrite").saveAsTable(
"mvp_filmes_imdb.default.catalogo_filmes")

spark.table("mvp_filmes_imdb.default.catalogo_filmes").count()

display(spark.table("mvp_filmes_imdb.default.frequencia_generos").groupBy("periodo").count())

In [0]:
filmes_para_grafico = spark.table("mvp_filmes_imdb.default.catalogo_filmes").withColumn("periodo",
F.when(F.col("startYear").cast("int").between(2000, 2009), "2000-2009")
.when(F.col("startYear").cast("int").between(2010, 2019), "2010-2019")
.when(F.col("startYear").cast("int").between(2020, 2026), "2020-2026"))
display(filmes_para_grafico.groupBy("periodo").count())
filmes_para_grafico = filmes_para_grafico.withColumn(
"averageRating", F.col("averageRating").cast("double"))
display(filmes_para_grafico)


display(spark.table("mvp_filmes_imdb.default.frequencia_generos"))

display(spark.table("mvp_filmes_imdb.default.frequencia_generos")
        .filter(F.col("genre").isin("Drama", "Adventure", "Action", "Comedy", "Sci-Fi")))

# Análise de filmes do IMDb

Este projeto analisou dados de títulos e avaliações do IMDb para selecionar filmes que possam compor o catálogo de uma loja de produtos culturais. Foram considerados apenas títulos classificados como filmes, lançados entre 2000 e 2026, com nota média igual ou superior a 7,0 e gênero informado.

Os filmes foram divididos em três períodos: 2000–2009, 2010–2019 e 2020–2026. Em cada período, foram selecionados os 100 filmes com maior número de votos, formando um catálogo de 300 filmes. Os dados de títulos e avaliações foram combinados e tratados no Databricks com PySpark. O catálogo final e a tabela de frequência dos gêneros foram salvos para consulta.

Também foi analisada a frequência dos gêneros em cada período. Quando um filme possui mais de um gênero, ele é contado uma vez em cada gênero ao qual pertence. Entre os 100 filmes selecionados de 2000–2009, Drama foi o gênero mais frequente, presente em 45 filmes. Em 2010–2019, Ação liderou, com 44 filmes. Em 2020–2026, Drama voltou a liderar, com 49 filmes.

Os resultados descrevem os filmes selecionados por estes critérios e permitem comparar o catálogo e seus gêneros entre os três períodos.